# Introduction

In [1]:
##Different Packages
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns 
from sklearn.preprocessing import OneHotEncoder
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from functionsFolder.config import text_data, biometric_data, adv_to_del, team_to_del, id, useless, index, to_keep
from functionsFolder.preProcessingAuto import preprocess_working_df
from functionsFolder.modelAutomation import get_features_and_target, clean_features, evaluate_models, prepare_features
from time import time

##Different Paths
HOME = r"C:\Users\Utilisateur\Desktop\Master ULB\Mémoire"
W_DB = r"\Database\Working db"
STACK = r"\Thesis - Code\database\Database Updates\Database stack"

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Utilisateur\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Utilisateur\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\Utilisateur\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


In [ ]:
base_df = pd.read_excel(HOME + STACK + r"\looping_df_1746884524.8275206.xlsx")
RECOVERY_OPP = HOME + W_DB + r"\Features\Team data\27-4-2025_team.xlsx"

## Preprocessing

In [ ]:
#add the mean of every position in the advance part and scouting evaluation part
base_df[to_keep['adv']] = base_df[to_keep['adv']].fillna(base_df.groupby('Pos_x_features')[to_keep['adv']].transform('median'))
base_df[to_keep['scouting_reports']] = base_df[to_keep['scouting_reports']].fillna(base_df.groupby('Pos_x_features')[to_keep['scouting_reports']].transform('median'))
processed_df = preprocess_working_df(base_df, recovery_file=RECOVERY_OPP)
#Get a copy of the dataset
working_df = processed_df.copy()
# Get initial features and targets
raw_features, target = get_features_and_target(working_df) #Get the features and target columns' name in separate lists


# Step 2: Prepare feature list (optionally re-adding 'adv' and 'scouting_reports')
include_subsets = ['per_40', 'team', 'opp', 'adv', 'scouting_reports']
features, included_types = prepare_features( #prepare the features' space
    base_features=raw_features,
    df=working_df,
    useless_list=useless, #list of features that we do not use 
    feature_dict=to_keep, #dictionnary of the features that we keep 
    include_keys=include_subsets #the keys to the dictionnary above
)
#Change the height from feet to inches 
working_df['Height_features'] = working_df['Height_features'].apply(
    lambda x: int(x.split('-')[0]) * 12 + int(x.split('-')[1])
)

#We keep only the players that where drafted after 2009 (60 players for 15 years = 900 max)
df = working_df[working_df['draft_season_features'] > 8].copy()
i = 1
df['med_tresh'] = (df[f'WS/48-{i}_target'] > 0).astype(int)

#We keep context variables in an list
keywords = ['team', 'opp', 'text']
filtered_list = [element for element in features if not any(keyword in element for keyword in keywords)]

team_to_avg = [
    col for col in test_idf.columns
    if ('team' in col or 'opp' in col) and 'text_' not in col and '%' not in col
]  #Creating a list with the columns to be divided

team_to_avg.remove('G_team_features_context') #Removing the number of games

test_idf[team_to_avg] = test_idf[team_to_avg].div(test_idf['G_team_features_context'], axis=0) #Dividing all the columns by the number of games

#df[features].to_excel('df_for_final.xlsx', index=False)

# Data Analysis

## Height

In [ ]:
plt.figure(figsize=(10, 6), dpi=360, facecolor='white') 

# NBA-inspired styling
nba_palette = ['#1D428A', '#C8102E', '#FDB927', '#000000', '#007A33']
court_color = '#F5F5F5'
main_color = nba_palette[1]
secondary_color = nba_palette[0] 

#Create the plot with enhanced parameters
ax = sns.kdeplot(df['Height_features']*0.0254, 
                fill=True, 
                color=main_color,
                alpha=0.8,
                linewidth=1,
                edgecolor=secondary_color)

# Add mean and median lines
mean_height = df['Height_features'].mean()*0.0254
median_height = df['Height_features'].median()*0.0254
plt.axvline(mean_height, color='black', linestyle='--', linewidth=1.5, alpha=0.7)
plt.axvline(median_height, color=secondary_color, linestyle='-', linewidth=1.5, alpha=0.7)

# Add annotations
plt.text(mean_height-0.08, ax.get_ylim()[1]*0.8, f'Mean: {mean_height:.2f}m', 
         fontsize=10, color='black')
plt.text(median_height+0.01, ax.get_ylim()[1]*0.9, f'Median: {median_height:.2f}m', 
         fontsize=10, color=secondary_color)

# Style enhancements
plt.title('Height Distribution for Drafted NBA Players (2009-2024)', 
          fontsize=14, fontweight='bold', pad=20, color='black')
plt.xlabel('Height (meters)', fontsize=12, labelpad=10)
plt.ylabel('Density', fontsize=12, labelpad=10)
plt.yticks([])

# Customize spines and grid
sns.despine(left=True)
plt.grid(axis='x', linestyle=':', alpha=0.4)


plt.tight_layout()
plt.show()

## Wordcloud

In [ ]:
word = r"C:\Users\Utilisateur\Desktop\Master ULB\Mémoire\Database\Working db\Features\Scouting report\Scouting reports - Copie (2).xlsx"
wordcloud = pd.read_excel(word, sheet_name='Scouting Report 2006-2025')

In [ ]:
from wordcloud import WordCloud, STOPWORDS
from nltk.corpus import stopwords
import re

# First, ensure you have NLTK stopwords
import nltk
nltk.download('stopwords')

text = wordcloud['Weakness'] + wordcloud['Strength']

## Step 1: Prepare Custom Stopwords
# Basic English stopwords
base_stopwords = set(stopwords.words('english'))

# Basketball-specific stopwords
basketball_stopwords = {
    'player', 'players', 'game', 'games', 'ability', 'can', 'one', 'will',
    'team', 'teams', 'play', 'plays', 'played', 'playing', 'nba', 'draft',
    'good', 'great', 'make', 'makes', 'like', 'get', 'gets', 'need', 'needs',
    'time', 'year', 'years', 'old', 'pro', 'professional', 'level', 'combine', 'season',
    'guard', 'shot', 'ball', 'rim', 'hes', 'per', 'doesnt'
}

# Combine with WordCloud's default stopwords
all_stopwords = base_stopwords.union(basketball_stopwords).union(STOPWORDS)

## Step 2: Text Processing Function
def process_text(text):
    # Remove special characters and numbers
    text = re.sub(r'[^a-zA-Z\s]', '', str(text))
    # Convert to lowercase
    text = text.lower()
    return text

## Step 3: Prepare Text Data
# Combine all scouting reports
all_text = ' '.join(text.apply(process_text).dropna())

## Step 4: Generate Word Cloud
wordcloud = WordCloud(
    width=1200,
    height=800,
    background_color='white',
    stopwords=all_stopwords,
    colormap='Reds',  # NBA red theme
    max_words=150,
    contour_width=3,
    contour_color='#1D428A',  # NBA blue
    collocations=False,  # Don't show word pairs
    prefer_horizontal=0.9,
    min_font_size=8,
    max_font_size=200,
    relative_scaling=0.5
).generate(all_text)

## Step 5: Visualization
plt.figure(figsize=(16, 10), facecolor='#F5F5F5')
plt.imshow(wordcloud.recolor(color_func=nba_color_func, random_state=3), interpolation='bilinear')
plt.axis('off')
plt.title('NBA Scouting Reports - Key Evaluation terms', 
         fontsize=20, pad=20, color='black', fontweight='bold')
plt.tight_layout(pad=0)
plt.show()

## Boxplot per 40

In [ ]:
per40_cols = ['per_40_PTS_features',
'per_40_AST_features',
'per_40_ORB_features',
'per_40_DRB_features',
'per_40_TRB_features',
'per_40_STL_features',
'per_40_BLK_features',
'per_40_TOV_features',
'per_40_PF_features',
'per_40_3P_features',
'per_40_3PA_features',
'per_40_2P_features',
'per_40_2PA_features',
'per_40_FT_features',
'per_40_FTA_features',
'advanced_OWS_features',
'advanced_DWS_features',
'advanced_WS_features',
'advanced_WS/40_features']

In [ ]:
plt.figure(figsize=(16, 10), facecolor='#FEFEFE', dpi=120)

# NBA color palette
primary_color = '#1D428A'  # Warriors blue
secondary_color = '#C8102E'  # Bulls red
tertiary_color = '#FDB927'  # Lakers gold

# Create custom palette - alternate colors for better differentiation
custom_palette = [primary_color if i%2==0 else secondary_color for i in range(len(per40_cols))]

# Create the boxplot
ax = sns.boxplot(data=df[per40_cols], 
                palette=custom_palette,
                width=0.7,
                linewidth=2,
                fliersize=4,
                boxprops=dict(alpha=0.9),
                medianprops=dict(color=tertiary_color, linewidth=2.5))

# Custom x-tick labels with abbreviations
stat_labels = ['PTS', 'AST', 'ORB', 'DRB', 'TRB', 'STL', 'BLK', 
               'TOV', 'PF', '3PM', '3PA', '2PM', '2PA', 'FTM', 'FTA', 
               'OWS', 'DWS', 'WS', 'WS/40']

# Add mean markers
for i, col in enumerate(per40_cols):
    mean_val = df[col].mean()
    ax.scatter(i, mean_val, color='white', edgecolor='black', 
               s=100, zorder=3, linewidth=1.5, marker='D')

# Styling enhancements
plt.title('NBA Per 40 Minute Statistics Distribution\n2009-2024 Drafted players', 
          fontsize=18, pad=20, color='black', fontweight='bold')
plt.xlabel('Statistical Categories', fontsize=14, labelpad=15)
plt.ylabel('Value per 40 Minutes', fontsize=14, labelpad=15)
plt.xticks(range(len(per40_cols)), stat_labels, 
           rotation=45, ha='right', fontsize=12)
plt.yticks(fontsize=12)

# Add grid and adjust spines
ax.set_axisbelow(True)
ax.grid(axis='y', linestyle=':', alpha=0.4, color='grey')
sns.despine(top=True, right=True, left=True)
ax.spines['bottom'].set_color(primary_color)
ax.spines['bottom'].set_linewidth(2)

# Add NBA-style watermark
plt.figtext(0.5, 0.15, "NBA SCOUTING DASHBOARD", 
            ha="center", fontsize=24, 
            color='#F5F5F5', alpha=0.2, fontweight='bold')


plt.tight_layout()
plt.show()

## Class distribution

In [ ]:
working_df['med_tresh']
# Compter le nombre d'occurrences de chaque classe dans 'med_tresh'
class_counts = working_df['med_tresh'].value_counts()

# Création du pie chart
plt.figure(figsize=(6, 6), dpi=250)
plt.pie(class_counts, labels=['Positive WS/48', 'Negative WS/48'], autopct='%1.1f%%', startangle=90,
        colors=[nba_palette[0], nba_palette[2]])
plt.axis('equal')  # Pour un cercle parfait
plt.show()

## Next